Código gerado por Claude versão XXXXXXX

colaraquioprompt

# Somador Carry-Lookahead Quântico de n qubits (Qiskit)

Implementacao do **somador carry-lookahead** de Draper-Kutin-Rains-Svore (2004), de **profundidade O(log n)**.

Versao *out-of-place*: leva `|a>|b>|0...0>` em `|a>|b>|a+b>`. Ao final, `a` e `b` sao preservados e todas as ancillas voltam a `|0>`.

Rode as celulas em ordem (Runtime > Run all).

## 1. Instalar o Qiskit

In [ ]:
!pip install qiskit qiskit-aer --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.7 MB/s eta 0:00:00


## 2. O somador carry-lookahead
A rede de carry tem 4 rodadas: **P** (propagacao de blocos), **G** (gera carries nas potencias de 2), **C** (preenche os carries intermediarios) e **P-inversa** (limpa as ancillas). Por isso a profundidade cresce com `log n`, e nao linearmente como no somador ripple-carry.

In [ ]:
import math, random
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator


def _ptree(n):
    """Indexa os qubits-ancilla de propagacao de bloco P[t][m] (t>=1)."""
    idx, k = {}, 0
    logn = int(math.floor(math.log2(n))) if n >= 1 else 0
    for t in range(1, logn + 1):
        for m in range(1, n // (2 ** t)):
            idx[(t, m)] = k; k += 1
    return idx, k, logn


def _carry_ccx(n, cmap, pmap, ancmap):
    """Lista dos Toffolis (CCX) da rede de carry-lookahead.
    Entrada: c[i]=generate g_i, p[i]=propagate. Saida: c[i]=carry de entrada do bit i+1."""
    Pidx, _, logn = _ptree(n)
    P = lambda t, m: pmap[m] if t == 0 else ancmap[(t, m)]
    g = []
    # Rodada P:  P[t][m] = P[t-1][2m] AND P[t-1][2m+1]
    for t in range(1, logn + 1):
        for m in range(1, n // (2 ** t)):
            g.append((P(t - 1, 2 * m), P(t - 1, 2 * m + 1), P(t, m)))
    # Rodada G: gera carries nos limites das potencias de 2
    for t in range(1, logn + 1):
        for m in range(0, n // (2 ** t)):
            g.append((P(t - 1, 2 * m + 1),
                      cmap[2 ** t * m + 2 ** (t - 1) - 1],
                      cmap[2 ** t * m + 2 ** t - 1]))
    # Rodada C: preenche os carries intermediarios
    if n >= 2:
        for t in range(int(math.floor(math.log2((2 * n) / 3.0))), 0, -1):
            for m in range(1, (n - 2 ** (t - 1)) // (2 ** t) + 1):
                g.append((P(t - 1, 2 * m),
                          cmap[2 ** t * m - 1],
                          cmap[2 ** t * m + 2 ** (t - 1) - 1]))
    # Rodada P-inversa: descomputa as ancillas de propagacao
    for t in range(logn, 0, -1):
        for m in range(1, n // (2 ** t)):
            g.append((P(t - 1, 2 * m), P(t - 1, 2 * m + 1), P(t, m)))
    return g


def somador_carry_lookahead(n):
    """Cria o circuito. Registradores: a(n), b(n), s(n+1)=resultado, c(n)=carry, panc=ancillas.
    Ao final: a e b preservados, c e panc voltam a |0>, e s = a + b (com bit de carry-out)."""
    a = QuantumRegister(n, 'a'); b = QuantumRegister(n, 'b')
    s = QuantumRegister(n + 1, 's'); c = QuantumRegister(n, 'c')
    Pidx, pk, logn = _ptree(n)
    regs = [a, b, s, c]
    panc = QuantumRegister(pk, 'panc') if pk > 0 else None
    if panc is not None: regs.append(panc)
    qc = QuantumCircuit(*regs)

    gi = lambda q: qc.find_bit(q).index
    cmap = [gi(c[i]) for i in range(n)]
    pmap = [gi(b[i]) for i in range(n)]
    ancmap = {k: gi(panc[v]) for k, v in Pidx.items()} if pk > 0 else {}
    cg = _carry_ccx(n, cmap, pmap, ancmap)

    for i in range(n): qc.ccx(a[i], b[i], c[i])         # 1) generate  g_i  -> c
    for i in range(n): qc.cx(a[i], b[i])                # 2) propagate p_i  -> b
    for x1, x2, tg in cg: qc.ccx(x1, x2, tg)            # 3) rede de carry-lookahead
    for i in range(n):                                  # 4) soma  s_i = p_i XOR c_i
        qc.cx(b[i], s[i])
        if i >= 1: qc.cx(c[i - 1], s[i])
    qc.cx(c[n - 1], s[n])                               #    carry-out -> bit mais alto
    for x1, x2, tg in reversed(cg): qc.ccx(x1, x2, tg)  # 5) desfaz a rede de carry
    for i in range(n): qc.cx(a[i], b[i])                # 6) desfaz propagate
    for i in range(n): qc.ccx(a[i], b[i], c[i])         # 7) desfaz generate
    return qc, (a, b, s, c)


def somar(x, y, n, shots=64):
    """Soma x + y (n bits cada) rodando o circuito no AerSimulator."""
    qc_add, (a, b, s, c) = somador_carry_lookahead(n)
    qc = QuantumCircuit(*qc_add.qregs)
    for i in range(n):
        if (x >> i) & 1: qc.x(a[i])
        if (y >> i) & 1: qc.x(b[i])
    qc.compose(qc_add, inplace=True)
    out = ClassicalRegister(n + 1, 'out'); qc.add_register(out)
    for i in range(n + 1): qc.measure(s[i], out[i])
    # method='matrix_product_state' mantem o estado de base exato e escala para n grande
    counts = AerSimulator(method='matrix_product_state').run(qc, shots=shots).result().get_counts()
    bits = max(counts, key=counts.get).replace(' ', '')  # char da esquerda = out[n] (MSB)
    return int(bits, 2)

## 3. Demonstracao rapida

In [ ]:
print("5  + 3  =", somar(5, 3, n=4))
print("13 + 9  =", somar(13, 9, n=4))
print("250 + 130 =", somar(250, 130, n=8))

# Ver o circuito para n pequeno:
qc, _ = somador_carry_lookahead(3)
print("\nn=3 -> qubits:", qc.num_qubits, "| profundidade:", qc.depth())
qc.draw("text")

5  + 3  = 8
13 + 9  = 22
250 + 130 = 380

n=3 -> qubits: 13 | profundidade: 10


»
a_0: ──■──────────────■────────────────────────■───────────────────────────»
       │              │                        │                           »
a_1: ──┼────■─────────┼────■───────────────────┼───────────────────────────»
       │    │         │    │                   │                           »
a_2: ──┼────┼────■────┼────┼────■──────────────┼───────────────────────────»
       │    │    │  ┌─┴─┐  │    │            ┌─┴─┐                         »
b_0: ──■────┼────┼──┤ X ├──┼────┼─────────■──┤ X ├─────────────────────────»
       │    │    │  └───┘┌─┴─┐  │         │  └───┘                         »
b_1: ──┼────■────┼───────┤ X ├──┼────■────┼─────────■──────────────────────»
       │    │    │       └───┘┌─┴─┐  │    │         │                      »
b_2: ──┼────┼────■────────────┤ X ├──┼────┼────■────┼─────────■─────────■──»
       │    │    │            └───┘  │  ┌─┴─┐  │    │         │         │  »
s_0: ──┼────┼────┼───────────────────┼──┤ X ├──┼────┼─────────┼─────────┼──»
       │    │    │                   │  └───┘  │  ┌─┴─┐┌───┐  │         │  »
s_1: ──┼────┼────┼───────────────────┼─────────┼──┤ X ├┤ X ├──┼─────────┼──»
       │    │    │                   │         │  └───┘└─┬─┘┌─┴─┐┌───┐  │  »
s_2: ──┼────┼────┼───────────────────┼─────────┼─────────┼──┤ X ├┤ X ├──┼──»
       │    │    │                   │         │  ┌───┐  │  └───┘└─┬─┘  │  »
s_3: ──┼────┼────┼───────────────────┼─────────┼──┤ X ├──┼─────────┼────┼──»
     ┌─┴─┐  │    │                   │         │  └─┬─┘  │         │    │  »
c_0: ┤ X ├──┼────┼───────────────────■─────────┼────┼────■─────────┼────┼──»
     └───┘┌─┴─┐  │                 ┌─┴─┐       │    │              │    │  »
c_1: ─────┤ X ├──┼─────────────────┤ X ├───────■────┼──────────────■────■──»
          └───┘┌─┴─┐               └───┘     ┌─┴─┐  │                 ┌─┴─┐»
c_2: ──────────┤ X ├─────────────────────────┤ X ├──■─────────────────┤ X ├»
               └───┘                         └───┘                    └───┘»
«                                   
«a_0: ─────────────────■────────────
«                      │            
«a_1: ────────────■────┼─────────■──
«                 │    │         │  
«a_2: ───────■────┼────┼────■────┼──
«            │    │    │    │    │  
«b_0: ───────┼────┼────■────┼────┼──
«            │  ┌─┴─┐  │    │    │  
«b_1: ──■────┼──┤ X ├──┼────┼────■──
«       │  ┌─┴─┐└───┘  │    │    │  
«b_2: ──┼──┤ X ├───────┼────■────┼──
«       │  └───┘       │    │    │  
«s_0: ──┼──────────────┼────┼────┼──
«       │              │    │    │  
«s_1: ──┼──────────────┼────┼────┼──
«       │              │    │    │  
«s_2: ──┼──────────────┼────┼────┼──
«       │              │    │    │  
«s_3: ──┼──────────────┼────┼────┼──
«       │            ┌─┴─┐  │    │  
«c_0: ──■────────────┤ X ├──┼────┼──
«     ┌─┴─┐          └───┘  │  ┌─┴─┐
«c_1: ┤ X ├─────────────────┼──┤ X ├
«     └───┘               ┌─┴─┐└───┘
«c_2: ────────────────────┤ X ├─────
«                         └───┘

## 4. Testes (verificar se o somador funciona)
Tres niveis de teste:
1. **Exaustivo no simulador quantico (Aer)** para n pequeno: roda o circuito de verdade e compara com `x+y`.
2. **Verificacao classica rapida** para n grande: o circuito so usa X/CX/CCX (reversivel classico), entao da pra simular bit-a-bit e testar n=16, 32... em milissegundos, checando tambem que `a`,`b` foram preservados e as ancillas zeraram.
3. **Profundidade**: mostra que ela cresce ~log n (carry-lookahead) e nao ~n.

In [ ]:
# ---- Teste 1: exaustivo no AerSimulator (simulacao quantica real) ----
print("Teste 1 - Aer (exaustivo):")
for n in [2, 3, 4]:
    ok = all(somar(x, y, n) == x + y for x in range(2**n) for y in range(2**n))
    print(f"  n={n}  ({4**n} casos): {'OK' if ok else 'FALHOU'}")

Teste 1 - Aer (exaustivo):
  n=2  (16 casos): OK
  n=3  (64 casos): OK
  n=4  (256 casos): OK


In [ ]:
# ---- Teste 2: verificacao classica rapida (n grande) ----
def _sim_reversivel(qc, init):
    bits = list(init)
    gi = lambda q: qc.find_bit(q).index
    for inst in qc.data:
        nm, qs = inst.operation.name, [gi(q) for q in inst.qubits]
        if   nm == "x":  bits[qs[0]] ^= 1
        elif nm == "cx": bits[qs[1]] ^= bits[qs[0]]
        elif nm == "ccx": bits[qs[2]] ^= (bits[qs[0]] & bits[qs[1]])
    return bits

def verifica(n, casos=None):
    qc, (a, b, s, c) = somador_carry_lookahead(n)
    gi = lambda q: qc.find_bit(q).index
    am=[gi(a[i]) for i in range(n)]; bm=[gi(b[i]) for i in range(n)]
    sm=[gi(s[i]) for i in range(n+1)]; cm=[gi(c[i]) for i in range(n)]
    pares = ([(x, y) for x in range(2**n) for y in range(2**n)] if casos is None
             else [(random.randrange(2**n), random.randrange(2**n)) for _ in range(casos)])
    for x, y in pares:
        init = [0]*qc.num_qubits
        for i in range(n): init[am[i]] = (x>>i)&1; init[bm[i]] = (y>>i)&1
        out = _sim_reversivel(qc, init)
        soma = sum(out[sm[i]]<<i for i in range(n+1))
        a_ok = sum(out[am[i]]<<i for i in range(n)) == x
        b_ok = sum(out[bm[i]]<<i for i in range(n)) == y
        anc_ok = all(out[cm[i]] == 0 for i in range(n))
        if not (soma == x+y and a_ok and b_ok and anc_ok):
            return False
    return True

print("Teste 2 - verificacao classica rapida:")
for n in range(1, 11):
    print(f"  n={n:2d} (exaustivo): {'OK' if verifica(n) else 'FALHOU'}")
for n in [16, 32, 64]:
    print(f"  n={n:2d} (500 aleatorios): {'OK' if verifica(n, casos=500) else 'FALHOU'}")

Teste 2 - verificacao classica rapida:
  n= 1 (exaustivo): OK
  n= 2 (exaustivo): OK
  n= 3 (exaustivo): OK
  n= 4 (exaustivo): OK
  n= 5 (exaustivo): OK
  n= 6 (exaustivo): OK
  n= 7 (exaustivo): OK
  n= 8 (exaustivo): OK
  n= 9 (exaustivo): OK
  n=10 (exaustivo): OK
  n=16 (500 aleatorios): OK
  n=32 (500 aleatorios): OK
  n=64 (500 aleatorios): OK


In [ ]:
# ---- Teste 3: profundidade ~ log n (a marca do carry-lookahead) ----
from qiskit import transpile
print("Teste 3 - profundidade vs n:")
print(f"  {'n':>4} | {'qubits':>6} | {'profundidade':>12}")
for n in [4, 8, 16, 32, 64]:
    qc, _ = somador_carry_lookahead(n)
    d = transpile(qc, basis_gates=["cx", "u3"]).depth()
    print(f"  {n:>4} | {qc.num_qubits:>6} | {d:>12}")

Teste 3 - profundidade vs n:
     n | qubits | profundidade
     4 |     18 |          112
     8 |     37 |          153
    16 |     76 |          194
    32 |    155 |          235
    64 |    314 |          276
